In [1]:
import pandas as pd

df = pd.read_csv("../data/processed/telco_churn_clean.csv")
df.shape

(7043, 21)

In [2]:
# customerID não é uma feature - é só identificador
# Churn é o target

X = df.drop(columns=["customerID", "Churn"])
y = df["Churn"].map({"No": 0, "Yes": 1})

print(X.shape, y.shape)
print(y.value_counts())

(7043, 19) (7043,)
Churn
0    5174
1    1869
Name: count, dtype: int64


In [3]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(X_train.shape, X_test.shape)
print(y_train.value_counts(normalize=True))
print(y_test.value_counts(normalize=True))

(5634, 19) (1409, 19)
Churn
0    0.734647
1    0.265353
Name: proportion, dtype: float64
Churn
0    0.734564
1    0.265436
Name: proportion, dtype: float64


In [4]:
numeric_features = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features = X.select_dtypes(include=["object"]).columns.tolist()

print("Numéricas:", numeric_features)
print("Categóricas:", categorical_features)


Numéricas: ['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges']
Categóricas: ['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod']


In [5]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
    ]
)

In [6]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import classification_report, roc_auc_score

dummy_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", DummyClassifier(strategy="most_frequent", random_state=42)),
])

dummy_pipeline.fit(X_train, y_train)

y_pred_dummy = dummy_pipeline.predict(X_test)
y_proba_dummy = dummy_pipeline.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred_dummy))
print("AUC-ROC:", roc_auc_score(y_test, y_proba_dummy))

              precision    recall  f1-score   support

           0       0.73      1.00      0.85      1035
           1       0.00      0.00      0.00       374

    accuracy                           0.73      1409
   macro avg       0.37      0.50      0.42      1409
weighted avg       0.54      0.73      0.62      1409

AUC-ROC: 0.5


c:\dev\tech-challenge-fase1\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\dev\tech-challenge-fase1\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\dev\tech-challenge-fase1\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


In [7]:
from sklearn.linear_model import LogisticRegression

logreg_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000, random_state=42)),
])

logreg_pipeline.fit(X_train, y_train)

y_pred_logreg = logreg_pipeline.predict(X_test)
y_proba_logreg = logreg_pipeline.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred_logreg, zero_division=0))
print("AUC-ROC:", roc_auc_score(y_test, y_proba_logreg))

              precision    recall  f1-score   support

           0       0.85      0.89      0.87      1035
           1       0.66      0.56      0.60       374

    accuracy                           0.81      1409
   macro avg       0.75      0.73      0.74      1409
weighted avg       0.80      0.81      0.80      1409

AUC-ROC: 0.8421349040274871


In [8]:
import mlflow
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
mlflow.set_tracking_uri(f"sqlite:///{PROJECT_ROOT / 'mlflow.db'}")
print("Tracking URI:", mlflow.get_tracking_uri())

Tracking URI: sqlite:///c:\dev\tech-challenge-fase1\mlflow.db


In [9]:
import mlflow

mlflow.set_experiment("churn-prediction-baselines")

with mlflow.start_run(run_name="dummy_classifier"):
    mlflow.log_param("strategy", "most_frequent")
    mlflow.log_param("model_type", "DummyClassifier")

    auc = roc_auc_score(y_test, y_proba_dummy)
    mlflow.log_metric("auc_roc", auc)
    mlflow.log_metric("accuracy", dummy_pipeline.score(X_test, y_test))

print("Run do Dummy registrada no MLflow.")

2026/06/14 18:20:55 INFO mlflow.tracking.fluent: Experiment with name 'churn-prediction-baselines' does not exist. Creating a new experiment.


Run do Dummy registrada no MLflow.


In [10]:
with mlflow.start_run(run_name="logistic_regression"):
    mlflow.log_param("model_type", "LogisticRegression")
    mlflow.log_param("max_iter", 1000)
    mlflow.log_param("random_state", 42)

    auc = roc_auc_score(y_test, y_proba_logreg)
    mlflow.log_metric("auc_roc", auc)
    mlflow.log_metric("accuracy", logreg_pipeline.score(X_test, y_test))
    mlflow.log_metric("f1_churn", 0.60)
    mlflow.log_metric("recall_churn", 0.56)
    mlflow.log_metric("precision_churn", 0.66)

print("Run da Regressão Logística registrada no MLflow.")


Run da Regressão Logística registrada no MLflow.
